# 06 — Mechanism-threshold calibration
Labeled set: monitoring-relevant (IVN + adjacent) vs irrelevant. Run on server. Output = score table to pick `mechanism_edge`.

In [ ]:
import os
import yaml
if os.getenv('NOTEBOOK_MOCK') == '1':
    print('=== MOCKED EMBEDDINGS (NOTEBOOK_MOCK=1, local run only, NOT calibrated) ===')
    import hashlib
    import numpy as np
    class MockModel:
        """Deterministic hash-based mock embedding."""
        def encode(self, texts, normalize_embeddings=True):
            """Hash each text to a fixed unit vector."""
            V = []
            for t in texts:
                h = hashlib.md5(t.encode()).digest()
                v = np.frombuffer(h, dtype=np.uint8).astype(float)
                v = v / max(np.linalg.norm(v), 1e-9)
                V.append(v)
            return np.array(V)
    m = MockModel()
else:
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer('zeroentropy/zembed-1-embedding', trust_remote_code=True)
    except Exception as e:
        raise RuntimeError(f'real model required for calibration, load failed: {e}')
def _lens_path():
    """Pick lens yaml path for repo-root or notebooks/ cwd."""
    for c in ('config/lenses/monitoring.yaml', '../config/lenses/monitoring.yaml'):
        if os.path.exists(c):
            return c
    return 'config/lenses/monitoring.yaml'
lens = yaml.safe_load(open(_lens_path()))
probes = [p for mech in lens['mechanisms'].values() for p in (mech.get('probes', []) if isinstance(mech, dict) else [])]
print(len(probes), 'probe phrases')

In [ ]:
def _thesis():
    """Read first thesis statement from ivn scope yaml."""
    import os
    for c in ('config/scopes/ivn.yaml', '../config/scopes/ivn.yaml'):
        if os.path.exists(c):
            txt = open(c).read()
            return txt.split('thesis_statements:')[1].split('\n')[1].strip(' -"\'')
    return 'deep learning for in-vehicle CAN intrusion detection'
THESIS = _thesis()
# Relevant set: REAL titles with verifiable sources (scores computed at runtime, never hardcoded)
POS = [
 ('rel-drift-survey', 'A Survey on Concept Drift Adaptation'),  # Springer: Gama et al., ACM Comput Surv 2014
 ('rel-calibration', 'On Calibration of Modern Neural Networks'),  # arXiv:1706.04599 Guo et al. 2017
 ('rel-ensemble', 'Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles'),  # arXiv:1712.06585 Lakshminarayanan et al. 2017
 ('rel-selective', 'Selective Classification for Deep Neural Networks'),  # arXiv:1705.08500 Geifman & El-Yaniv 2017
 ('rel-canids', 'OTIDS: A Novel Intrusion Detection System for In-vehicle Network by Using Remote Frame'),  # IEEE Lee et al. 2017 CAN IDS
 ('rel-nist', 'Guide to Intrusion Detection and Prevention Systems'),  # NIST SP 800-94 Scarfone & Mell
]
# Controls: all REAL, genuinely irrelevant, each CONTROL marked with verifiable link
NEG = [
 ('neg-hair', 'HairCLIP: Design Your Hair by Text and Reference Image'),  # CONTROL hair editing; arXiv:2112.05142
 ('neg-protein', 'Highly accurate protein structure prediction with AlphaFold'),  # CONTROL protein folding; Nature 2021 Jumper et al. doi:10.1038/s41586-021-03819-2
 ('neg-legal', 'CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review'),  # CONTROL legal contracts; arXiv:2103.06268
 ('neg-imagenet', 'Deep Residual Learning for Image Recognition'),  # CONTROL pure-accuracy vision, no monitoring; arXiv:1512.03385
 ('neg-go', 'Mastering the Game of Go with Deep Neural Networks and Tree Search'),  # CONTROL board game, no monitoring; Nature 2016 Silver et al. doi:10.1038/nature16961
]
print(len(POS), 'pos,', len(NEG), 'neg')

In [ ]:
import numpy as np
P = m.encode(['query: ' + p for p in probes], normalize_embeddings=True)
D = m.encode(['passage: ' + t for _, t in POS + NEG], normalize_embeddings=True)
T = m.encode(['query: ' + THESIS], normalize_embeddings=True)
mech = (D @ P.T).max(axis=1)
thes = (D @ T.T).ravel()
print(f"{'id':<14}{'thesis':>8}{'mech':>8}  label")
for (pid, _), th, me in zip(POS + NEG, thes, mech):
    lab = 'POS' if pid in dict(POS) else 'NEG'
    print(f'{pid:<14}{th:>8.3f}{me:>8.3f}  {lab}')
pos_mech = mech[:len(POS)]
neg_mech = mech[len(POS):]
print(f'\ncomputed min-POS-mech={pos_mech.min():.3f} max-NEG-mech={neg_mech.max():.3f}')
print('propose mechanism_edge between max-NEG-mech and min-POS-mech above (overlap = no clean split, record best-effort)')
print(f"computed admits at edge 0.37: {(pos_mech >= 0.37).sum()}/{len(pos_mech)} pos + {(neg_mech >= 0.37).sum()}/{len(neg_mech)} neg")

In [ ]:
import os, sys
for _c in ('..', '.'):
    """Ensure repo root on path for pipeline import."""
    if os.path.isdir(os.path.join(_c, 'pipeline')) and _c not in sys.path:
        sys.path.insert(0, _c)
from pipeline import rank as rankmod
def _fake_score(thesis, texts, model):
    """Canned thesis scores: 0.85 thesis crowd, 0.793 mechanism-only crowd."""
    return [0.793 if t.startswith('mech-only') else 0.85 for t in texts]
def _fake_fit(items, mechanisms, model):
    """Canned mechanism fits: high for mech-only tail, low otherwise."""
    return [('m', 0.50) if (i.get('title') or '').startswith('mech-only') else ('', 0.10) for i in items]
_os, _of = rankmod.score, rankmod.mechanism_fit
rankmod.score, rankmod.mechanism_fit = _fake_score, _fake_fit
try:
    _items = [{'title': f'thesis-{k}'} for k in range(22)] + [{'title': f'mech-only-{k}'} for k in range(3)]
    _ranked, _ = rankmod.rank_with_stats(_items, 't', None, threshold=0.80, keep=20, lens='monitoring', mechanism_threshold=0.37)
    assert len(_ranked) <= 20, f'quota exceeded keep: {len(_ranked)}'
    _q = [d for d in _ranked if d['selection_reason'] == 'mechanism-quota']
    assert len(_q) == 3, f'quota swap failed: {len(_q)}'
    assert all(d['score'] == 0.793 for d in _q), 'mechanism-only must survive 0.793 thesis crowd'
    print(f'quota assert ok: len={len(_ranked)}<=20, mechanism-quota={len(_q)}, 0.793-crowd survives')
finally:
    rankmod.score, rankmod.mechanism_fit = _os, _of